# MILESTONE 3

Install Libraries

In [ ]:
!pip install sentence-transformers
!pip install faiss-cpu
!pip install neo4j
!pip install groq
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 13.7 MB/s eta 0:00:00


Load Enron Emails

In [ ]:
import pandas as pd

emails = pd.read_csv("enron_emails_processed.csv")

emails = emails.dropna()

emails.head()

FileNotFoundError: [Errno 2] No such file or directory: 'enron_emails_processed.csv'

Generate Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

email_texts = emails["body"].tolist()

embeddings = model.encode(email_texts)


#check vector size
print(embeddings.shape)

Create FAISS Vector Database

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Total emails indexed:", index.ntotal)

Test Semantic Search

In [ ]:
query = "energy trading discussion"

query_embedding = model.encode([query])

D, I = index.search(query_embedding, k=5)

results = emails.iloc[I[0]]

results

Connect to Neo4j Graph

In [ ]:
from neo4j import GraphDatabase

uri = "bolt://localhost:7687"
username = "neo4j"
password = "YOUR_PASSWORD"

driver = GraphDatabase.driver(uri, auth=(username, password))

#Test connection.
def test_connection(tx):
    result = tx.run("MATCH (n) RETURN count(n) as total")
    return result.single()["total"]

with driver.session() as session:
    nodes = session.execute_read(test_connection)
    print("Total Nodes:", nodes)

Graph Retrieval

In [ ]:
def get_graph_context(name):

    query = """
    MATCH (p:PERSON {name:$name})-[r]-(n)
    RETURN p.name as person, type(r) as relation, n.name as neighbor
    LIMIT 10
    """

    with driver.session() as session:
        result = session.run(query, name=name)
        return [record.data() for record in result]

    #get_graph_context("phillip k")

Build Hybrid Retrieval

In [ ]:
def retrieve_context(question):

    query_embedding = model.encode([question])

    D, I = index.search(query_embedding, k=3)

    email_results = emails.iloc[I[0]]["body"].tolist()

    graph_results = get_graph_context("phillip k")

    return email_results, graph_results

    #Test
    emails_ctx, graph_ctx = retrieve_context(
    "Who did Phillip K communicate with about trading?"
)

print(emails_ctx)
print(graph_ctx)

Connect LLM (Groq)

In [ ]:
from groq import Groq

client = Groq(api_key="YOUR_GROQ_KEY")

RAG Prompt

In [ ]:
def generate_answer(question):

    emails_ctx, graph_ctx = retrieve_context(question)

    context = f"""
Emails:
{emails_ctx}

Graph:
{graph_ctx}
"""

    response = client.chat.completions.create(

        model="llama3-70b-8192",

        messages=[
            {"role": "system", "content": "You are an enterprise email analyst."},
            {"role": "user", "content": question + "\nContext:\n" + context}
        ]
    )

    return response.choices[0].message.content

Test RAG

In [ ]:
question = "Who communicated most with Phillip K?"

answer = generate_answer(question)

print(answer)